# 02 — OCR Political Content

Objetivo:

> Usar os OCR pickles dos telejornais para extrair informação política: temas, candidatos, partidos, diferenças RTP vs TVI e timelines de blocos políticos.

Versão limpa:
- As antigas secções 1, 2 e 3 foram reduzidas para uma única célula de preparação.
- Foram removidos os plots exploratórios de top words/bigrams.
- A partir da secção 4, mantive a tua análise.


## Preparação mínima dos dados

Esta célula substitui as antigas secções 1, 2 e 3.

Ela apenas:
1. importa bibliotecas;
2. carrega `ocr_long`;
3. limpa texto;
4. cria as colunas necessárias para as secções seguintes:

```python
clean_text
tokens_content
n_words_content
```

As análises começam a sério na secção 4.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "features"
OUTPUT_DIR = BASE_DIR / "outputs_ocr_02"
OUTPUT_DIR.mkdir(exist_ok=True)

NORMALIZED_PATH = BASE_DIR / "outputs_ocr_01" / "ocr_long_normalized.csv"

print("BASE_DIR:", BASE_DIR.resolve())
print("FEATURES_DIR exists:", FEATURES_DIR.exists())
print("NORMALIZED_PATH exists:", NORMALIZED_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


# ---------- Loading helpers ----------

def parse_ocr_filename(path):
    path = Path(path)
    stem = path.stem.replace("_ocr", "")
    parts = stem.split("_")

    channel = parts[1] if len(parts) >= 2 else None
    month = parts[2] if len(parts) >= 3 else None
    day = parts[3] if len(parts) >= 4 else None

    return {
        "file": path.name,
        "channel": channel,
        "month": month,
        "day": day,
        "date_label": f"{month}_{day}" if month and day else None
    }

def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def extract_frame_number(frame_value, fallback_index):
    if isinstance(frame_value, (str, Path)):
        nums = re.findall(r"\d+", Path(str(frame_value)).stem)
        if nums:
            return int(nums[-1])
        return int(fallback_index)

    try:
        return int(float(frame_value))
    except Exception:
        return int(fallback_index)

def extract_text(det):
    if isinstance(det, dict):
        for key in ["text", "Text", "word", "value", "label"]:
            if key in det:
                return str(det[key])
    if isinstance(det, str):
        return det
    return None

def extract_conf(det):
    if isinstance(det, dict):
        for key in ["conf", "confidence", "score", "prob", "probability"]:
            if key in det:
                return safe_float(det[key])
    return np.nan

def normalize_ocr_df(df, file_name):
    meta = parse_ocr_filename(file_name)
    rows = []

    frame_col = "Frame" if "Frame" in df.columns else df.columns[0]
    ocr_col = "OCR" if "OCR" in df.columns else None

    if ocr_col is None:
        raise ValueError(f"Não encontrei coluna OCR em {file_name}. Colunas: {list(df.columns)}")

    for idx, row in df.iterrows():
        frame_original = row[frame_col]
        frame_number = extract_frame_number(frame_original, idx)
        detections = row[ocr_col]

        if not isinstance(detections, (list, tuple, np.ndarray)):
            continue

        for det in detections:
            text = extract_text(det)
            if text is None or len(str(text).strip()) == 0:
                continue

            rows.append({
                "file": file_name,
                "channel": meta["channel"],
                "date_label": meta["date_label"],
                "frame": frame_number,
                "frame_original": str(frame_original),
                "text": str(text),
                "confidence": extract_conf(det),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["second"] = out["frame"]
    out["minute"] = (out["second"] // 60).astype(int)
    return out


# ---------- Load normalized OCR ----------

if NORMALIZED_PATH.exists():
    ocr_long = pd.read_csv(NORMALIZED_PATH)
else:
    print("Normalized OCR not found. Loading raw OCR pickles...")
    ocr_files = sorted(FEATURES_DIR.glob("*_ocr.pkl"))
    parts = []

    for f in ocr_files:
        df = pd.read_pickle(f)
        norm = normalize_ocr_df(df, f.name)
        print(f.name, "->", len(norm), "detections")
        parts.append(norm)

    ocr_long = pd.concat(parts, ignore_index=True)

# Ensure expected types.
ocr_long["text"] = ocr_long["text"].astype(str)
ocr_long["frame"] = pd.to_numeric(ocr_long["frame"], errors="coerce")
ocr_long["minute"] = pd.to_numeric(ocr_long["minute"], errors="coerce").astype("Int64")


# ---------- Text cleaning helpers ----------

STOPWORDS_PT = {
    "de","a","o","e","que","do","da","em","um","uma","para","com","não","os","as","no","na","por",
    "se","ao","dos","das","mais","como","é","foi","são","ser","tem","também","ou","à","às","nos",
    "nas","sobre","entre","até","sem","já","lhe","ele","ela","eles","elas","sua","seu","suas","seus",
    "este","esta","estes","estas","isso","isto","há","vai","ter","mas","muito","muita","muitos","muitas",
    "porque","quando","onde","quem","qual","quais","todo","toda","todos","todas","num","numa","pelo","pela",
    "pelos","pelas","aos","ainda","só","era","foram","será","serem","nosso","nossa","seja","forma"
}

STRUCTURAL_WORDS = {
    "telejornal", "jornal", "nacional", "direto", "direita", "esquerda", "tvi", "rtp", "sic",
    "notícias", "noticia", "noticias", "edição", "especial", "última", "hora", "minuto",
    "portugal", "portuguesa", "português", "portugueses", "www", "pt"
}

def clean_text_pt(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-záàâãéèêíóôõúç0-9\s]", " ", text)
    text = re.sub(r"\b(b|br)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_pt(text, remove_stopwords=True, remove_structural=True, min_len=3):
    tokens = clean_text_pt(text).split()
    tokens = [t for t in tokens if len(t) >= min_len]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]

    if remove_structural:
        tokens = [t for t in tokens if t not in STRUCTURAL_WORDS]

    return tokens

ocr_long["clean_text"] = ocr_long["text"].apply(clean_text_pt)
ocr_long["tokens_content"] = ocr_long["clean_text"].apply(lambda x: tokenize_pt(x))
ocr_long["n_words_content"] = ocr_long["tokens_content"].apply(len)

print("ocr_long ready:", ocr_long.shape)
print("Files:", ocr_long["file"].nunique())
print("Channels:", sorted(ocr_long["channel"].dropna().unique()))
print("Dates:", sorted(ocr_long["date_label"].dropna().unique()))


## 4. Dicionário de candidatos, partidos e temas políticos

Podes editar estes dicionários à medida que descobrires mais nomes/termos relevantes.


In [ ]:
import re
import pandas as pd
from pathlib import Path

# candidate_party.pkl is used only as reference here.
candidate_party_path = FEATURES_DIR / "candidate_party.pkl"

if candidate_party_path.exists():
    candidate_party = pd.read_pickle(candidate_party_path)
    print("candidate_party:", candidate_party.shape)
    display(candidate_party)
else:
    candidate_party = None
    print("candidate_party.pkl não encontrado.")


# ------------------------------------------------------------
# Candidate aliases
# ------------------------------------------------------------
# Note:
# Some aliases are descriptive expressions used in TV news, not only names.
# Be careful with generic terms such as "seguro" or "almirante", which may create false positives.

CANDIDATE_ALIASES = {
    "André Ventura": [
        "andré ventura",
        "andre ventura",
        "ventura",
        "líder do chega",
        "lider do chega"
    ],

    "Cotrim Figueiredo": [
        "cotrim",
        "cotrim de figueiredo",
        "cotrim figueiredo",
        "joão cotrim de figueiredo",
        "joao cotrim de figueiredo"
    ],

    "Luís Marques Mendes": [
        "marques mendes",
        "luís marques mendes",
        "luis marques mendes"
    ],

    "Henrique Gouveia e Melo": [
        "gouveia e melo",
        "gouveia melo",
        "henrique gouveia e melo",
        "almirante gouveia e melo"
        # "almirante" can be added, but it is more generic.
    ],

    "António José Seguro": [
        "antónio josé seguro",
        "antonio jose seguro",
        "josé seguro",
        "jose seguro",
        "antónio seguro",
        "antonio seguro",
        # "seguro" alone is intentionally excluded because it can be a common word.
    ],

    "António Filipe": [
        "antónio filipe",
        "antonio filipe"
    ],

    "Catarina Martins": [
        "catarina martins",
        "ex coordenadora do bloco",
        "ex-coordenadora do bloco",
        "ex lider do bloco",
        "ex-lider do bloco"
    ],

    "Jorge Pinto": [
        "jorge pinto"
        # "pinto" alone is intentionally excluded because it can create false positives.
    ],
}


# ------------------------------------------------------------
# Party aliases
# ------------------------------------------------------------
# Short acronyms such as "PS", "IL", "BE" can create OCR false positives.
# We keep them, but interpret results carefully.

PARTY_ALIASES = {
    "CHEGA": [
        "chega",
        "partido chega"
    ],

    "IL": [
        "il",
        "iniciativa liberal",
        "liberais"
    ],

    "PSD": [
        "psd",
        "ad",
        "aliança democrática",
        "alianca democratica",
        "partido social democrata",
        "sociais democratas"
    ],

    "PS": [
        "ps",
        "partido socialista",
        "socialistas"
    ],

    "PCP/CDU": [
        "pcp",
        "cdu",
        "partido comunista",
        "comunistas"
    ],

    "BE": [
        "be",
        "bloco de esquerda",
    ],

    "LIVRE": [
        "livre",
        "partido livre"
    ],

    "CDS": [
        "cds",
        "cds pp",
        "cds-pp",
        "centro democrático social",
    ],
}


# ------------------------------------------------------------
# Topic/theme aliases
# ------------------------------------------------------------

THEME_ALIASES = {
    "Eleições/Campanha": [
        "presidenciais",
        "presidencial",
        "eleições",
        "eleicao",
        "eleições presidenciais",
        "eleicoes presidenciais",
        "candidato",
        "candidata",
        "candidatos",
        "candidatas",
        "candidatura",
        "candidaturas",
        "campanha",
        "abstenção",
        "abstencao",
        "urna",
        "urnas",
        "eleitor",
        "eleitores",
        "debate",
        "debates"
    ],

    "Sondagens": [
        "sondagem",
        "sondagens",
        "barómetro",
        "barometro",
        "voto",
        "votos",
        "intenção de voto",
        "intencao de voto",
        "intenções de voto",
        "intencoes de voto",
        "projecção",
        "projeccao",
        "projeção",
        "projecao"
    ],

    "Governo/Partidos": [
        "governo",
        "primeiro ministro",
        "primeira ministra",
        "montenegro",
        "luís montenegro",
        "luis montenegro",
        "parlamento",
        "assembleia",
        "partido",
        "partidos",
        "oposição",
        "oposicao"
    ],

    "Saúde": [
        "saúde",
        "saude",
        "sns",
        "hospital",
        "hospitais",
        "médico",
        "medico",
        "médicos",
        "medicos",
        "enfermeiro",
        "enfermeiros",
        "urgência",
        "urgencias",
        "urgência",
        "inem",
        "listas de espera",
        "lista de espera",
        "ambulância",
        "ambulancia"
    ],

    "Economia": [
        "economia",
        "inflação",
        "inflacao",
        "preços",
        "precos",
        "imposto",
        "impostos",
        "irs",
        "orçamento",
        "orcamento",
        "salário",
        "salario",
        "salários",
        "salarios",
        "rendimento",
        "pib",
        "juros",
        "bce"
    ],

    "Habitação": [
        "habitação",
        "habitacao",
        "casa",
        "casas",
        "arrendamento",
        "renda",
        "rendas",
        "senhorio",
        "senhorios",
        "inquilino",
        "inquilinos",
        "crédito habitação",
        "credito habitacao"
    ],

    "Educação": [
        "educação",
        "educacao",
        "escola",
        "escolas",
        "professor",
        "professores",
        "aluno",
        "alunos",
        "ensino",
        "aulas",
        "creche",
        "creches"
    ],

    "Justiça/Segurança": [
        "justiça",
        "justica",
        "tribunal",
        "tribunais",
        "polícia",
        "policia",
        "pj",
        "psp",
        "gnr",
        "segurança",
        "seguranca",
        "crime",
        "crimes",
        "corrupção",
        "corrupcao",
        "pgr",
        "ministério público",
        "ministerio publico"
    ],

    "Internacional": [
        "ucrânia",
        "ucrania",
        "rússia",
        "russia",
        "guerra",
        "israel",
        "gaza",
        "palestina",
        "trump",
        "eua",
        "estados unidos",
        "europa",
        "bruxelas",
        "união europeia",
        "uniao europeia"
    ],

    "Greves/Trabalho": [
        "greve",
        "greves",
        "sindicato",
        "sindicatos",
        "trabalhador",
        "trabalhadores",
        "protesto",
        "protestos",
        "manifestação",
        "manifestacao",
        "manifestantes",
        "patrões",
        "patroes"
    ],
}


# ------------------------------------------------------------
# Alias matching functions
# ------------------------------------------------------------

def normalize_alias(alias):
    """
    Applies the same text normalization used for OCR text.
    """
    return clean_text_pt(alias)


def contains_alias(text, aliases):
    """
    Returns True if any alias appears in the OCR text.

    Uses word-boundary-like regex to avoid partial matches.
    Example: "ps" should not match inside another longer word.
    """
    if not isinstance(text, str):
        return False

    text = clean_text_pt(text)

    for alias in aliases:
        alias = normalize_alias(alias)

        if not alias:
            continue

        pattern = r"(?<!\w)" + re.escape(alias) + r"(?!\w)"

        if re.search(pattern, text):
            return True

    return False


print("N candidates:", len(CANDIDATE_ALIASES))
print("N parties:", len(PARTY_ALIASES))
print("N themes:", len(THEME_ALIASES))

## 5. Menções globais a candidatos, partidos e temas

Contamos em quantas deteções OCR aparece cada entidade/tema.

Nota: isto conta deteções OCR, não pessoas reais no vídeo.


In [ ]:
def count_mentions(df, alias_dict, label_col_name):
    rows = []
    total_detections = len(df)
    total_words = df["n_words_content"].sum()

    for label, aliases in alias_dict.items():
        mask = df["clean_text"].apply(lambda t: contains_alias(t, aliases))
        detections = int(mask.sum())
        frames = int(df.loc[mask, ["file", "frame"]].drop_duplicates().shape[0])
        files = int(df.loc[mask, "file"].nunique())

        rows.append({
            label_col_name: label,
            "detections": detections,
            "frames": frames,
            "files": files,
            "per_1000_detections": 1000 * detections / max(total_detections, 1),
            "per_1000_words": 1000 * detections / max(total_words, 1),
        })

    return pd.DataFrame(rows).sort_values("detections", ascending=False)

candidate_counts = count_mentions(ocr_long, CANDIDATE_ALIASES, "candidate")
party_counts = count_mentions(ocr_long, PARTY_ALIASES, "party")
theme_counts = count_mentions(ocr_long, THEME_ALIASES, "theme")

display(Markdown("### Candidatos"))
display(candidate_counts)

display(Markdown("### Partidos"))
display(party_counts)

display(Markdown("### Temas"))
display(theme_counts)

candidate_counts.to_csv(OUTPUT_DIR / "candidate_mentions_global.csv", index=False)
party_counts.to_csv(OUTPUT_DIR / "party_mentions_global.csv", index=False)
theme_counts.to_csv(OUTPUT_DIR / "theme_mentions_global.csv", index=False)


In [ ]:
# Plots principais
plt.figure(figsize=(10, 4))
plt.bar(candidate_counts["candidate"], candidate_counts["detections"])
plt.title("Menções OCR por candidato — global")
plt.ylabel("Nº deteções")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(theme_counts["theme"], theme_counts["detections"])
plt.title("Menções OCR por tema — global")
plt.ylabel("Nº deteções")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Menções por telejornal

Aqui calculamos os temas/candidatos por ficheiro, canal e data.


In [ ]:
def mention_table_by_file(df, alias_dict, entity_type):
    rows = []

    for (file, channel, date_label), group in df.groupby(["file", "channel", "date_label"]):
        total_detections = len(group)
        total_words = group["n_words_content"].sum()

        for label, aliases in alias_dict.items():
            mask = group["clean_text"].apply(lambda t: contains_alias(t, aliases))
            detections = int(mask.sum())

            rows.append({
                "file": file,
                "channel": channel,
                "date_label": date_label,
                "entity_type": entity_type,
                "entity": label,
                "detections": detections,
                "per_1000_detections": 1000 * detections / max(total_detections, 1),
                "per_1000_words": 1000 * detections / max(total_words, 1),
            })

    return pd.DataFrame(rows)

candidate_by_file = mention_table_by_file(ocr_long, CANDIDATE_ALIASES, "candidate")
party_by_file = mention_table_by_file(ocr_long, PARTY_ALIASES, "party")
theme_by_file = mention_table_by_file(ocr_long, THEME_ALIASES, "theme")

display(theme_by_file.sort_values("detections", ascending=False).head(30))
display(candidate_by_file.sort_values("detections", ascending=False).head(30))

candidate_by_file.to_csv(OUTPUT_DIR / "candidate_mentions_by_file.csv", index=False)
party_by_file.to_csv(OUTPUT_DIR / "party_mentions_by_file.csv", index=False)
theme_by_file.to_csv(OUTPUT_DIR / "theme_mentions_by_file.csv", index=False)


## 7. Heatmaps: temas e candidatos por telejornal

Estas tabelas ajudam a ver rapidamente onde cada tema/candidato aparece mais.


In [ ]:
MONTH_ORDER = {
    "Nov": 1,
    "Dec": 2,
    "Jan": 3
}

CHANNEL_ORDER = {
    "RTP": 1,
    "TVI": 2
}

def date_label_to_sort(date_label):
    """
    Converts labels like Nov_10, Dec_02, Jan_13 into sortable numeric values.
    """
    month, day = str(date_label).split("_")
    return MONTH_ORDER.get(month, 999) * 100 + int(day)


def plot_heatmap_from_table(table, entity_col, value_col, title):
    table = table.copy()

    # Create chronological sorting columns
    table["date_sort"] = table["date_label"].apply(date_label_to_sort)
    table["channel_sort"] = table["channel"].map(CHANNEL_ORDER).fillna(999)

    # Correct column order: Nov -> Dec -> Jan, and RTP -> TVI within each date
    col_order = (
        table[["date_label", "channel", "date_sort", "channel_sort"]]
        .drop_duplicates()
        .sort_values(["date_sort", "channel_sort"])
    )

    ordered_columns = pd.MultiIndex.from_frame(
        col_order[["date_label", "channel"]]
    )

    # Pivot table
    pivot = table.pivot_table(
        index=entity_col,
        columns=["date_label", "channel"],
        values=value_col,
        aggfunc="sum",
        fill_value=0
    )

    # Reorder columns chronologically
    pivot = pivot.reindex(columns=ordered_columns, fill_value=0)

    display(pivot)

    plt.figure(figsize=(14, max(4, 0.45 * len(pivot))))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label=value_col)

    plt.yticks(
        range(len(pivot.index)),
        pivot.index
    )

    plt.xticks(
        range(len(pivot.columns)),
        [f"{date}-{channel}" for date, channel in pivot.columns],
        rotation=45,
        ha="right"
    )

    plt.title(title)
    plt.tight_layout()
    plt.show()

    return pivot


theme_heatmap = plot_heatmap_from_table(
    theme_by_file,
    entity_col="entity",
    value_col="per_1000_words",
    title="Temas por telejornal — normalizado por 1000 palavras OCR"
)

candidate_heatmap = plot_heatmap_from_table(
    candidate_by_file,
    entity_col="entity",
    value_col="per_1000_words",
    title="Candidatos por telejornal — normalizado por 1000 palavras OCR"
)

In [ ]:
def summarize_entity_by_file_minutes(df, alias_dict, entity_type):
    rows = []

    for entity, aliases in alias_dict.items():
        temp = df.copy()

        temp["has_entity"] = temp["clean_text"].apply(
            lambda text: contains_alias(text, aliases)
        )

        hits = temp[temp["has_entity"]].copy()

        if len(hits) == 0:
            continue

        grouped = (
            hits.groupby(["file", "channel", "date_label"])
            .agg(
                detections=("text", "size"),
                frames=("frame", "nunique"),
                minutes=("minute", "nunique"),
                first_minute=("minute", "min"),
                last_minute=("minute", "max")
            )
            .reset_index()
        )

        grouped["entity"] = entity
        grouped["entity_type"] = entity_type

        rows.append(grouped)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


candidate_by_file_minutes = summarize_entity_by_file_minutes(
    ocr_long,
    CANDIDATE_ALIASES,
    "candidate"
)

display(
    candidate_by_file_minutes
    .sort_values(["minutes", "frames", "detections"], ascending=False)
    .head(30)
)

## 8. RTP vs TVI nas mesmas datas

Comparação direta entre canais para as mesmas datas.

A métrica principal é normalizada por 1000 palavras OCR para evitar que telejornais com mais texto ganhem sempre.


In [ ]:
# ============================================================
# 8. RTP vs TVI — comparação no mesmo dia
# ============================================================

MONTH_ORDER = {
    "Nov": 1,
    "Dec": 2,
    "Jan": 3
}

CHANNEL_ORDER = ["RTP", "TVI"]

def date_label_to_sort(date_label):
    month, day = str(date_label).split("_")
    return MONTH_ORDER.get(month, 999) * 100 + int(day)


# Ficheiros suspeitos / contaminados
SUSPECT_FILES = [
    "Telejornal_TVI_Dec_2_ocr.pkl"
]

# Filtrar tabelas já calculadas
theme_by_file_analysis = theme_by_file.copy()
candidate_by_file_analysis = candidate_by_file.copy()

if "file" in theme_by_file_analysis.columns:
    theme_by_file_analysis = theme_by_file_analysis[
        ~theme_by_file_analysis["file"].isin(SUSPECT_FILES)
    ].copy()

if "file" in candidate_by_file_analysis.columns:
    candidate_by_file_analysis = candidate_by_file_analysis[
        ~candidate_by_file_analysis["file"].isin(SUSPECT_FILES)
    ].copy()


# Temas: RTP vs TVI por data
theme_channel_date = (
    theme_by_file_analysis
    .groupby(["date_label", "channel", "entity"])
    .agg(
        detections=("detections", "sum"),
        per_1000_detections=("per_1000_detections", "sum"),
        per_1000_words=("per_1000_words", "sum")
    )
    .reset_index()
)

# Candidatos: RTP vs TVI por data
candidate_channel_date = (
    candidate_by_file_analysis
    .groupby(["date_label", "channel", "entity"])
    .agg(
        detections=("detections", "sum"),
        per_1000_detections=("per_1000_detections", "sum"),
        per_1000_words=("per_1000_words", "sum")
    )
    .reset_index()
)

display(theme_channel_date.head())
display(candidate_channel_date.head())

In [ ]:
# Escolher apenas datas onde existem os dois canais: RTP e TVI

date_channel_counts = (
    theme_channel_date[["date_label", "channel"]]
    .drop_duplicates()
    .groupby("date_label")["channel"]
    .nunique()
)

paired_dates = date_channel_counts[date_channel_counts == 2].index.tolist()
paired_dates = sorted(paired_dates, key=date_label_to_sort)

print("Datas com RTP e TVI disponíveis:")
print(paired_dates)

chosen_date = paired_dates[0]

print("chosen_date =", chosen_date)

In [ ]:
def plot_rtp_vs_tvi_for_date(table, chosen_date, title_prefix, value_col="per_1000_words"):
    data_date = table[table["date_label"] == chosen_date].copy()

    pivot = data_date.pivot_table(
        index="entity",
        columns="channel",
        values=value_col,
        aggfunc="sum",
        fill_value=0
    )

    # Garantir sempre ordem RTP, TVI
    pivot = pivot.reindex(columns=CHANNEL_ORDER, fill_value=0)

    # Remover linhas com zero nos dois canais
    pivot = pivot[pivot.sum(axis=1) > 0]

    # Ordenar por presença total
    pivot["total"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("total", ascending=False).drop(columns="total")

    display(pivot)

    pivot.plot(kind="bar", figsize=(11, 4))
    plt.title(f"{title_prefix} — {chosen_date}")
    plt.ylabel("Menções por 1000 palavras OCR")
    plt.xlabel("")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return pivot


pivot_theme_date = plot_rtp_vs_tvi_for_date(
    theme_channel_date,
    chosen_date=chosen_date,
    title_prefix="RTP vs TVI — temas no OCR"
)

pivot_candidate_date = plot_rtp_vs_tvi_for_date(
    candidate_channel_date,
    chosen_date=chosen_date,
    title_prefix="RTP vs TVI — candidatos no OCR"
)

In [ ]:
for date in paired_dates:
    print("=" * 80)
    print("Data:", date)

    plot_rtp_vs_tvi_for_date(
        theme_channel_date,
        chosen_date=date,
        title_prefix="RTP vs TVI — temas no OCR"
    )

    plot_rtp_vs_tvi_for_date(
        candidate_channel_date,
        chosen_date=date,
        title_prefix="RTP vs TVI — candidatos no OCR"
    )

## 9. Diferenças RTP - TVI por tema/candidato

Valores positivos indicam maior presença normalizada na RTP.  
Valores negativos indicam maior presença normalizada na TVI.


In [ ]:
def channel_difference_table(channel_date_table):
    pivot = channel_date_table.pivot_table(
        index=["date_label", "entity"],
        columns="channel",
        values="per_1000_words",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    if "RTP" not in pivot.columns:
        pivot["RTP"] = 0
    if "TVI" not in pivot.columns:
        pivot["TVI"] = 0

    pivot["RTP_minus_TVI"] = pivot["RTP"] - pivot["TVI"]
    pivot["abs_difference"] = pivot["RTP_minus_TVI"].abs()
    return pivot.sort_values("abs_difference", ascending=False)

theme_diff = channel_difference_table(theme_channel_date)
candidate_diff = channel_difference_table(candidate_channel_date)

display(Markdown("### Maiores diferenças por tema"))
display(theme_diff.head(30))

display(Markdown("### Maiores diferenças por candidato"))
display(candidate_diff.head(30))

theme_diff.to_csv(OUTPUT_DIR / "rtp_tvi_theme_differences.csv", index=False)
candidate_diff.to_csv(OUTPUT_DIR / "rtp_tvi_candidate_differences.csv", index=False)


## 10. Timeline política dentro de um telejornal

Escolhemos automaticamente o telejornal com mais menções políticas e vemos em que minutos aparecem os principais temas.


In [ ]:
# Criar flags de temas/candidatos por deteção OCR
for theme, aliases in THEME_ALIASES.items():
    col = "theme_" + re.sub(r"[^a-zA-Z0-9]+", "_", theme).strip("_")
    ocr_long[col] = ocr_long["clean_text"].apply(lambda t: contains_alias(t, aliases))

for candidate, aliases in CANDIDATE_ALIASES.items():
    col = "cand_" + re.sub(r"[^a-zA-Z0-9]+", "_", candidate).strip("_")
    ocr_long[col] = ocr_long["clean_text"].apply(lambda t: contains_alias(t, aliases))

theme_cols = [c for c in ocr_long.columns if c.startswith("theme_")]
candidate_cols = [c for c in ocr_long.columns if c.startswith("cand_")]

ocr_long["n_theme_hits"] = ocr_long[theme_cols].sum(axis=1)
ocr_long["n_candidate_hits"] = ocr_long[candidate_cols].sum(axis=1)
ocr_long["n_political_hits"] = ocr_long["n_theme_hits"] + ocr_long["n_candidate_hits"]

political_by_file = ocr_long.groupby("file")["n_political_hits"].sum().sort_values(ascending=False)
display(political_by_file.head(10))

chosen_file = political_by_file.index[0]
print("chosen_file =", chosen_file)


In [ ]:
timeline_cols = ["file", "minute", "n_political_hits", "n_theme_hits", "n_candidate_hits"]

timeline = ocr_long[ocr_long["file"] == chosen_file].groupby(["file", "minute"]).agg(
    n_detections=("text", "size"),
    n_words=("n_words_content", "sum"),
    n_theme_hits=("n_theme_hits", "sum"),
    n_candidate_hits=("n_candidate_hits", "sum"),
    n_political_hits=("n_political_hits", "sum"),
    text_concat=("clean_text", lambda s: " ".join(s))
).reset_index()

display(timeline.sort_values("n_political_hits", ascending=False).head(15))

plt.figure(figsize=(13, 4))
plt.plot(timeline["minute"], timeline["n_political_hits"], marker="o")
plt.title(f"Menções políticas OCR por minuto — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Nº hits políticos")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 4))
plt.plot(timeline["minute"], timeline["n_theme_hits"], marker="o", label="Temas")
plt.plot(timeline["minute"], timeline["n_candidate_hits"], marker="o", label="Candidatos")
plt.title(f"Temas vs candidatos no OCR por minuto — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Nº hits")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 10.1 Timeline de temas específicos

Aqui podes escolher temas/candidatos específicos.


In [ ]:
# Edita esta lista para testar outros termos.
selected_themes = ["Eleições/Campanha", "Sondagens", "Governo/Partidos"]
selected_candidates = ["André Ventura", "Henrique Gouveia e Melo", "Cotrim Figueiredo", "Luís Marques Mendes"]

plt.figure(figsize=(13, 5))

for theme in selected_themes:
    col = "theme_" + re.sub(r"[^a-zA-Z0-9]+", "_", theme).strip("_")
    if col in ocr_long.columns:
        temp = ocr_long[ocr_long["file"] == chosen_file].groupby("minute")[col].sum()
        plt.plot(temp.index, temp.values, marker="o", label=theme)

plt.title(f"Timeline de temas selecionados — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Menções OCR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))

for candidate in selected_candidates:
    col = "cand_" + re.sub(r"[^a-zA-Z0-9]+", "_", candidate).strip("_")
    if col in ocr_long.columns:
        temp = ocr_long[ocr_long["file"] == chosen_file].groupby("minute")[col].sum()
        plt.plot(temp.index, temp.values, marker="o", label=candidate)

plt.title(f"Timeline de candidatos selecionados — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Menções OCR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Minutos mais interessantes para validação manual

Estes são bons candidatos para ires depois buscar frames e mostrar exemplos visuais.


In [ ]:
interesting_minutes = timeline.sort_values("n_political_hits", ascending=False).head(15).copy()

# reduzir texto para leitura
interesting_minutes["text_preview"] = interesting_minutes["text_concat"].str.slice(0, 350)

display(interesting_minutes[[
    "file", "minute", "n_detections", "n_words",
    "n_theme_hits", "n_candidate_hits", "n_political_hits", "text_preview"
]])

interesting_minutes.to_csv(OUTPUT_DIR / "interesting_political_minutes.csv", index=False)


## 12. Coocorrência de candidatos no mesmo minuto

Isto pode indicar blocos em que vários candidatos são comparados, por exemplo em sondagens/debates.


In [ ]:
candidate_minute_rows = []

for (file, minute), group in ocr_long.groupby(["file", "minute"]):
    present = []
    for candidate in CANDIDATE_ALIASES:
        col = "cand_" + re.sub(r"[^a-zA-Z0-9]+", "_", candidate).strip("_")
        if col in group.columns and group[col].sum() > 0:
            present.append(candidate)

    if len(present) >= 2:
        candidate_minute_rows.append({
            "file": file,
            "minute": minute,
            "candidates_present": ", ".join(present),
            "n_candidates": len(present),
            "text_preview": " ".join(group["clean_text"]).strip()[:300]
        })

candidate_cooccurrence = pd.DataFrame(candidate_minute_rows).sort_values("n_candidates", ascending=False)

display(candidate_cooccurrence.head(30))
candidate_cooccurrence.to_csv(OUTPUT_DIR / "candidate_cooccurrence_by_minute.csv", index=False)


## 13. Conclusões automáticas rápidas

Esta célula gera frases iniciais que podes adaptar.


In [ ]:
display(Markdown("## Draft de conclusões"))

total_detections = len(ocr_long)
n_files = ocr_long["file"].nunique()
n_dates = ocr_long["date_label"].nunique()
channels = ", ".join(sorted(ocr_long["channel"].dropna().unique()))

top_candidate = candidate_counts.iloc[0]
top_theme = theme_counts.iloc[0]
top_file = political_by_file.index[0]
top_file_hits = int(political_by_file.iloc[0])

summary_lines = [
    f"Foram analisados {n_files} ficheiros OCR, de {channels}, cobrindo {n_dates} datas.",
    f"A tabela normalizada contém {total_detections:,} deteções OCR.",
    f"O tema mais frequente no OCR foi '{top_theme['theme']}', com {int(top_theme['detections'])} deteções.",
    f"O candidato mais mencionado no OCR foi '{top_candidate['candidate']}', com {int(top_candidate['detections'])} deteções.",
    f"O telejornal com maior número de hits políticos foi {top_file}, com {top_file_hits} hits.",
    "As métricas normalizadas por 1000 palavras OCR são importantes para comparar RTP e TVI de forma mais justa.",
    "Os minutos com mais hits políticos são bons candidatos para validação manual com frames e para cruzamento posterior com speech."
]

for line in summary_lines:
    print("-", line)

with open(OUTPUT_DIR / "draft_conclusions.txt", "w", encoding="utf-8") as f:
    for line in summary_lines:
        f.write("- " + line + "\n")


## 14. Outputs gerados

Ficheiros guardados em `outputs_ocr_02/`:

- `top_words_global.csv`
- `top_bigrams_global.csv`
- `candidate_mentions_global.csv`
- `party_mentions_global.csv`
- `theme_mentions_global.csv`
- `candidate_mentions_by_file.csv`
- `party_mentions_by_file.csv`
- `theme_mentions_by_file.csv`
- `rtp_tvi_theme_differences.csv`
- `rtp_tvi_candidate_differences.csv`
- `interesting_political_minutes.csv`
- `candidate_cooccurrence_by_minute.csv`
- `draft_conclusions.txt`
